# 05 – Phân loại (Học có giám sát)

Notebook này minh họa việc huấn luyện các mô hình phân loại (logistic regression, random forest, optional XGBoost) sử dụng dữ liệu **`cluster_input.parquet`**.  Mã xử lý nằm trong `src/models/supervised.py` và tập lệnh `scripts/run_modeling.py`.  Chúng ta sẽ tải dữ liệu, chuẩn bị đặc trưng, huấn luyện mô hình, đánh giá và trình bày kết quả.

*Lưu ý: chạy `scripts/run_clustering.py` trước khi mở notebook này để tạo `cluster_input.parquet` (RFM + nhãn phân cụm).*

In [ ]:
# imports and cấu hình đường dẫn
import os
import sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
from src.utils.config import load_config
from src.models import supervised
from src.evaluation import metrics

# plotting
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# load cấu hình và dữ liệu
cfg = load_config(os.path.join(ROOT, "configs", "params.yaml"))
processed_dir = os.path.join(ROOT, cfg["paths"]["processed_dir"])
input_path = os.path.join(processed_dir, "cluster_input.parquet")
df = pd.read_parquet(input_path)

print(df.shape)
df.head()

In [ ]:
# tách X, y và thực hiện train/test

# chọn cột mục tiêu từ config
mdl_cfg = cfg.get("modeling", {})
target_col = mdl_cfg.get("target", "target")

X, y = supervised.prepare_features(df, target_col=target_col)
X_train, X_test, y_train, y_test = supervised.split_data(X, y, test_size=mdl_cfg.get("test_size", 0.2))

print("số đặc trưng:", X.shape)
print("phân phối nhãn:\n", y.value_counts())

In [ ]:
# huấn luyện và đánh giá
models = supervised.train_models(X_train, y_train, algorithms=mdl_cfg.get("algorithms"))
metrics_df = supervised.evaluate_models(models, X_test, y_test)
metrics_df

In [ ]:
# xác định mô hình tốt nhất và vẽ ma trận nhầm lẫn
best = supervised.select_best_model(metrics_df)
best_model = models[best]

from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap='Blues')
disp.figure_.suptitle(f"Ma trận nhầm lẫn ({best})")
plt.show()

# vẽ tầm quan trọng đặc trưng
def plot_feat_importance(model, feature_names):
    imp = supervised.feature_importance(model, feature_names)
    if imp.empty:
        return
    imp.head(20).sort_values().plot(kind='barh', figsize=(8,6))
    plt.title(f"Tầm quan trọng đặc trưng ({best})")
    plt.show()

plot_feat_importance(best_model, list(X.columns))